In [1]:
# %%
import numpy as np

NON1_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_stft_non-speech_1.npy"
NON2_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_stft_non-speech_2.npy"
SPEECH_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_stft_speech.npy"

X_non1 = np.load(NON1_PATH, mmap_mode="r")
X_non2 = np.load(NON2_PATH, mmap_mode="r")
X_speech = np.load(SPEECH_PATH, mmap_mode="r")

print("Non1 :", X_non1.shape)
print("Non2 :", X_non2.shape)
print("Speech:", X_speech.shape)

Non1 : (1094364, 257)
Non2 : (168714, 257)
Speech: (1284459, 257)


In [2]:
# %%
# %%
X_non = np.concatenate([
    X_non1[:, :257],
    X_non2[:, :257]
], axis=0)

X_speech = X_speech[:, :257]

print("Non total:", X_non.shape)
print("Speech:", X_speech.shape)

X = np.vstack([X_non, X_speech]).astype(np.float32)

y = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])

print("X:", X.shape)
print("Label distribution:", np.bincount(y))

Non total: (1263078, 257)
Speech: (1284459, 257)
X: (2547537, 257)
Label distribution: [1263078 1284459]


In [3]:
# %%
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (2038029, 257)
Test : (509508, 257)


In [4]:
# %%
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

rf = RandomForestClassifier(
    n_estimators=80,
    max_depth=15,
    min_samples_leaf=10,
    n_jobs=4,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("=== RANDOM FOREST ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

=== RANDOM FOREST ===
Accuracy : 0.9513903608971792
Precision: 0.9434622629614204
Recall   : 0.9611899163850957
F1-score : 0.9522435891255681
AUC      : 0.9881082861772492


In [5]:
# %%
from xgboost import XGBClassifier

xgb = XGBClassifier(
    tree_method="hist",
    device="cuda",          # dùng GPU
    max_depth=10,
    n_estimators=300,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

print("=== XGBOOST (GPU) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_prob))

/home/quochuy/miniconda3/envs/audio_ml/lib/python3.10/site-packages/xgboost/core.py:774: UserWarning: [22:50:50] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


=== XGBOOST (GPU) ===
Accuracy : 0.9868304324956625
Precision: 0.9861830138519658
Recall   : 0.9877185743425253
F1-score : 0.9869501968167037
AUC      : 0.999091565440687


In [ ]:
# %%
import joblib

# Lưu Random Forest
joblib.dump(rf, "rf_stft.pkl")

# Lưu XGBoost
joblib.dump(xgb, "xgb_stft.pkl")

print("Models saved successfully!")

Models saved successfully!


: 